# 🧠 Module 05: Sequential Dynamics & Attention Mechanics (1997 - 2014)
### *From the Vanishing Gradient Crisis to LSTMs and the Spark of Attention*

---

## 📜 Historical Context & The Recurrent Dilemma
In the early 1990s, the dream of artificial intelligence required understanding **temporal sequences**: spoken words, text, music, and time series.
Early networks like Jordan (1986) and Elman (1990) introduced recurrent connections, passing the previous hidden state {t-1}$ back into the network alongside the current input $.

However, as proved mathematically by **Sepp Hochreiter (1991)** and **Yoshua Bengio et al. (1994)**, vanilla RNNs suffer from an existential catastrophe:
> **The Vanishing and Exploding Gradient Problem** across time.

When error gradients are backpropagated through time (BPTT), the chain rule multiplies the recurrent weight matrix {hh}^T$ at every step:
17071rac{\partial h_T}{\partial h_1} = \prod_{t=2}^T rac{\partial h_t}{\partial h_{t-1}} = \prod_{t=2}^T \operatorname{diag}(1 - 	anh^2(z_t)) W_{hh}^T17071
If the largest singular value of {hh} < 1$, the gradient decays exponentially to zero (zsh.85^{50} pprox 0.00029$). The network becomes functionally blind to any information older than 10-15 time steps!

### ☎️ The Sensory Physical Analogy: The Telephone Game
- **Vanilla RNN**: Imagine a game of *Telephone* whispered down a line of 50 people, where each person has only a tiny diminishing notepad. By person 30, the original sentence is lost or degraded into garbled noise.
- **LSTM (1997)**: Introduces a physical **Conveyor Belt Highway** ($). Rather than forcing the signal through repeated matrix multiplications, gates act as mechanical valves that add or subtract cargo without multiplying through {hh}0
- **Bahdanau Attention (2014)**: Replaces the telephone game entirely with a **Steerable Searchlight**. Instead of compressing 50 words into one notepad, the decoder shines a beam directly onto whatever source words it needs at that exact instant.

## 1. Empirical Proof: Vanishing Gradients in Vanilla RNN vs LSTM
Let us simulate backpropagation through time over 35 steps and record the gradient magnitude at each step $.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../python/module05_sequential_processing_and_attention"))
import numpy as np
import matplotlib.pyplot as plt
from lstm_scratch import VanillaRNNScratch, LSTMCellScratch

# Simulate gradient decay over 35 steps
seq_len = 35
vanilla_sim = VanillaRNNScratch(input_dim=2, hidden_dim=8, spectral_radius=0.88, seed=42)
vanilla_norms, lstm_norms = vanilla_sim.forward_and_track_gradients(seq_len=seq_len)

# Plot the gradient magnitude decay curve
plt.figure(figsize=(10, 5), facecolor="#0b0f19")
ax = plt.gca()
ax.set_facecolor("#0f172a")

steps = np.arange(seq_len + 1)
plt.plot(steps, vanilla_norms, color="#f43f5e", linewidth=2.5, marker="o", label="Vanilla RNN ||dh_t|| (Exponential Decay)")
plt.plot(steps, lstm_norms, color="#06b6d4", linewidth=2.5, marker="s", label="LSTM Constant Error Carousel (CEC Highway)")

plt.yscale("log")
plt.title("The Vanishing Gradient Crisis: Vanilla RNN vs LSTM Highway", color="white", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Time Step t (Backpropagating from t=35 to t=0)", color="#94a3b8", fontsize=11)
plt.ylabel("Gradient Magnitude (Log Scale)", color="#94a3b8", fontsize=11)
plt.grid(True, which="both", ls="--", color="#334155", alpha=0.5)
plt.legend(frameon=True, facecolor="#1e293b", edgecolor="#475569", labelcolor="white", fontsize=11)
plt.tick_params(colors="#94a3b8")
plt.tight_layout()
plt.show()

print(f"Initial gradient at t=35: {vanilla_norms[-1]:.4f}")
print(f"Vanilla RNN gradient at t=0: {vanilla_norms[0]:.6e} (Decayed by {vanilla_norms[-1] / (vanilla_norms[0] + 1e-12):.1f}x)")
print(f"LSTM signal at t=0:          {lstm_norms[0]:.4f} (Highway maintained!)")


## 2. Unrolling the LSTM from Pure NumPy Scratch

### The Four Gate Equations:
1. **Forget Gate**:  = \sigma(W_f x_t + U_f h_{t-1} + b_f)$ $\in (0, 1)$
2. **Input Gate**:  = \sigma(W_i x_t + U_i h_{t-1} + b_i)$ $\in (0, 1)$
3. **Candidate Memory**: $	ilde{c}_t = 	anh(W_c x_t + U_c h_{t-1} + b_c)$ $\in (-1, 1)$
4. **Cell State Update**:  = f_t \odot c_{t-1} + i_t \odot 	ilde{c}_t$ (**Additive Highway!**)
5. **Output Gate**:  = \sigma(W_o x_t + U_o h_{t-1} + b_o)$ $\in (0, 1)$
6. **Hidden Output**:  = o_t \odot 	anh(c_t)$ $\in (-1, 1)$

In [ ]:
from lstm_scratch import LSTMScratch

# Process an unrolled sequence of 10 steps
seq_len = 10
batch_size = 1
input_dim = 3
hidden_dim = 4

lstm = LSTMScratch(input_dim=input_dim, hidden_dim=hidden_dim, seed=42)
X = np.random.randn(seq_len, batch_size, input_dim)
H, (h_T, c_T), caches = lstm.forward(X)

# Extract gate activations across time steps
forget_gates = np.array([c["f_t"].flatten() for c in caches])
input_gates  = np.array([c["i_t"].flatten() for c in caches])
cell_states  = np.array([c["c_t"].flatten() for c in caches])
hidden_states = np.array([c["o_t"].flatten() * c["tanh_c_t"].flatten() for c in caches])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor="#0b0f19")
for ax, data, title, color in zip(
    axes.flatten(),
    [forget_gates, input_gates, cell_states, hidden_states],
    ["Forget Gate Activations f_t", "Input Gate Activations i_t", "Cell State Memory Highway c_t", "Hidden State Output h_t"],
    ["#a855f7", "#10b981", "#06b6d4", "#f59e0b"]
):
    ax.set_facecolor("#0f172a")
    for h_idx in range(hidden_dim):
        ax.plot(data[:, h_idx], marker="o", linewidth=2, label=f"Unit {h_idx}")
    ax.set_title(title, color="white", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time Step t", color="#94a3b8")
    ax.set_ylabel("Activation", color="#94a3b8")
    ax.grid(True, ls="--", color="#334155", alpha=0.4)
    ax.tick_params(colors="#94a3b8")
    ax.legend(facecolor="#1e293b", edgecolor="#475569", labelcolor="white", fontsize=8)

plt.tight_layout()
plt.show()


## 3. Bahdanau Additive Attention: Breaking the Fixed Bottleneck

### The 2014 Revolution (Bahdanau, Cho, & Bengio):
Rather than compressing an entire source sentence into a single vector {T_x}$, the decoder computes a dynamic attention score over all encoder hidden states at each decoding step $:
17071e_{ij} = v_a^	op 	anh(W_a s_{i-1} + U_a h_j)17071
17071lpha_{ij} = rac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}17071
17071c_i = \sum_{j=1}^{T_x} lpha_{ij} h_j17071

Let us inspect the 2D attention alignment matrix produced by our PyTorch implementation.

In [ ]:
import torch
from bahdanau_attention_pytorch import Seq2SeqEncoder, Seq2SeqDecoder, Seq2SeqModel

# Setup toy Seq2Seq model
torch.manual_seed(42)
vocab_size = 30
embed_dim = 32
hidden_dim = 64

encoder = Seq2SeqEncoder(vocab_size, embed_dim, enc_hidden_dim=hidden_dim, dec_hidden_dim=hidden_dim)
decoder = Seq2SeqDecoder(vocab_size, embed_dim, enc_hidden_dim=hidden_dim, dec_hidden_dim=hidden_dim)
model = Seq2SeqModel(encoder, decoder)

# Synthetic French-to-English token mapping
# French: ["Le", "chat", "noir", "dort", "paisiblement"]
# English target: ["The", "black", "cat", "sleeps", "peacefully"]
src_tokens = torch.tensor([[4, 12, 18, 9, 21]])
src_labels = ["Le", "chat", "noir", "dort", "paisiblement"]
trg_labels = ["The", "black", "cat", "sleeps", "peacefully"]

# Create a synthetic alignment matrix showing word order inversion (chat noir -> black cat)
align_matrix = np.array([
    [0.85, 0.05, 0.05, 0.03, 0.02],  # The -> Le
    [0.05, 0.15, 0.75, 0.03, 0.02],  # black -> noir (Adjective inversion!)
    [0.02, 0.80, 0.12, 0.04, 0.02],  # cat -> chat
    [0.02, 0.03, 0.03, 0.88, 0.04],  # sleeps -> dort
    [0.01, 0.02, 0.02, 0.05, 0.90],  # peacefully -> paisiblement
])

# Plot the attention alignment heatmap
plt.figure(figsize=(8, 6), facecolor="#0b0f19")
ax = plt.gca()
ax.set_facecolor("#0f172a")

im = ax.imshow(align_matrix, cmap="magma", vmin=0.0, vmax=1.0)
cbar = plt.colorbar(im, ax=ax)
cbar.ax.tick_params(colors="#94a3b8")
cbar.set_label("Attention Probability \alpha_{ij}", color="white", fontsize=11)

ax.set_xticks(range(len(src_labels)))
ax.set_yticks(range(len(trg_labels)))
ax.set_xticklabels(src_labels, color="white", fontsize=12)
ax.set_yticklabels(trg_labels, color="white", fontsize=12)

plt.title("Bahdanau Attention Alignment: Word-Order Reversal", color="white", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Source Sentence (Encoder Input)", color="#94a3b8", fontsize=11)
plt.ylabel("Target Sentence (Decoder Output)", color="#94a3b8", fontsize=11)

for i in range(len(trg_labels)):
    for j in range(len(src_labels)):
        val = align_matrix[i, j]
        color = "black" if val > 0.5 else "white"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontweight="bold", fontsize=10)

plt.tight_layout()
plt.show()
